In [1]:
# Step 1: Install dependencies & set Hugging Face token
%pip install -q "datasets>=2.19.0" "huggingface_hub>=0.24"
import os
import getpass

# 直接设置Hugging Face token，跳过登录界面
hf_token = getpass.getpass("Paste your Hugging Face token: ")
os.environ['HF_TOKEN'] = hf_token
os.environ['HUGGINGFACE_HUB_TOKEN'] = hf_token

print("Hugging Face token set successfully!")

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Paste your Hugging Face token:  ········


Hugging Face token set successfully!


In [2]:
# Step 2: Load FLARE-FINRED test set and preprocess
from datasets import load_dataset, Dataset

# FLARE-FINRED dataset
print("加载 FLARE-FINRED 数据集...")
ds_raw = load_dataset("ChanceFocus/flare-finred", split="test")
print(f"成功加载! 样本数: {len(ds_raw)}, 列: {ds_raw.column_names}")

# 从数据集内容中提取所有可能的关系类型
RELATION_TYPES = [
    'legal_form', 'publisher', 'owner_of', 'employer', 'manufacturer', 
    'position_held', 'chairperson', 'industry', 'business_division', 'creator',
    'original_broadcaster', 'chief_executive_officer', 'location_of_formation', 
    'operator', 'owned_by', 'founded_by', 'parent_organization', 'member_of',
    'product_or_material_produced', 'brand', 'headquarters_location',
    'director_/_manager', 'distribution_format', 'distributed_by', 'platform',
    'currency', 'subsidiary', 'stock_exchange', 'developer'
]

def parse_finred_answer(answer_list):
    """解析答案格式，将字符串列表转换为结构化triplet"""
    triplets = []
    if answer_list and isinstance(answer_list, list):
        for triplet_str in answer_list:
            if isinstance(triplet_str, str) and ';' in triplet_str:
                parts = [p.strip() for p in triplet_str.split(';')]
                if len(parts) == 3:
                    triplets.append({
                        "head": parts[0],
                        "tail": parts[1],
                        "relation": parts[2]
                    })
    return triplets

def _map_row(x):
    text = x.get("text") or x.get("sentence") or ""
    answer = x.get("answer", [])
    triplets = parse_finred_answer(answer)
    return {
        "text": text,
        "answer": answer,
        "triplets": triplets,
        "relation_types": RELATION_TYPES
    }

ds = Dataset.from_list([_map_row(r) for r in ds_raw])
bad = [i for i, r in enumerate(ds) if not r["text"]]
print(f"空文本样本数: {len(bad)}")
assert len(bad) == 0, "发现空文本样本。"

# 显示第一个样本作为验证
print("\n第一个样本示例:")
print(f"文本: {ds[0]['text'][:200]}...")
print(f"三元组: {ds[0]['triplets']}")

加载 FLARE-FINRED 数据集...
成功加载! 样本数: 1068, 列: ['id', 'query', 'text', 'answer', 'label']
空文本样本数: 0

第一个样本示例:
文本: Wednesday, July 8, 2015 10:30AM IST (5:00AM GMT) Rimini Street Comment on Oracle Litigation Las Vegas, United States Rimini Street, Inc., the leading independent provider of enterprise software suppor...
三元组: []


In [3]:
# Step 3: Install dependencies, configure OpenAI, and record experiment metadata
%pip install -q "openai==1.40.2" "httpx==0.27.2" "httpcore==1.0.5" \
               "pandas>=2.2.2" "tqdm>=4.66.4" "requests>=2.31.0"

import os, getpass, json, time, platform
from importlib.metadata import version, PackageNotFoundError
import requests

# o3-mini配置
MODEL = "o3-mini"
BASE_URL = "https://api.openai.com/v1"

api_key = os.getenv("OPENAI_API_KEY") or os.getenv("API_KEY")
if not api_key:
    api_key = getpass.getpass("Paste your OpenAI API key: ")
os.environ["OPENAI_API_KEY"] = api_key

# 文件命名
dataset_name = "flare_finred"
run_tag = f"{dataset_name}_{MODEL.replace('-', '_')}"
save_dir = "/content"
pred_path = f"{save_dir}/{run_tag}_predictions.csv"
meta_path = f"{save_dir}/{run_tag}_metadata.json"

def ver(pkg: str) -> str:
    try:
        return version(pkg)
    except PackageNotFoundError:
        return "not-installed"

meta = {
    "dataset": "ChanceFocus/flare-finred",
    "split": "test",
    "relation_types": RELATION_TYPES,
    "model": MODEL,
    "openai_sdk": ver("openai"),
    "httpx": ver("httpx"),
    "httpcore": ver("httpcore"),
    "datasets_version": ver("datasets"),
    "pandas": ver("pandas"),
    "tqdm": ver("tqdm"),
    "time_utc": time.strftime("%Y-%m-%d %H:%M:%S", time.gmtime()),
    "python": platform.python_version(),
    "base_url": BASE_URL,
    "note": "o3-mini evaluation on FLARE-FINRED (Financial Relation Extraction)"
}

os.makedirs(save_dir, exist_ok=True)
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print("Meta saved ->", meta_path)
print("MODEL:", MODEL, "| BASE_URL:", BASE_URL)

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Paste your OpenAI API key:  ········


Meta saved -> /content/flare_finred_o3_mini_metadata.json
MODEL: o3-mini | BASE_URL: https://api.openai.com/v1


In [4]:
# Step 4: FINRED 关系抽取推理代码
import json, os, re, time
import pandas as pd
from tqdm import tqdm
import requests

run_tag = f"{dataset_name}_{MODEL.replace('-', '_')}"
pred_path = f"{save_dir}/{run_tag}_predictions.csv"
err_path = f"{save_dir}/{run_tag}_errors.csv"
print(f"预测文件: {pred_path}")
print(f"错误文件: {err_path}")

def _strip_code_fences(s: str) -> str:
    s = s.strip()
    if s.startswith("```"):
        s = re.sub(r"^```[a-zA-Z0-9_-]*\s*", "", s)
        s = re.sub(r"\s*```$", "", s)
    return s.strip()

def _make_finred_prompt(text: str):
    """为FINRED任务创建提示词"""
    relations_str = ', '.join(RELATION_TYPES)
    return (
        "Task: Extract relation triplets from the following financial text.\n\n"
        f"Text: {text}\n\n"
        "Instructions:\n"
        f"1. Identify all pairs of entities that have a relationship from this list: {relations_str}\n"
        "2. For each relationship, output a triplet in the format: 'head ; tail ; relation'\n"
        "3. Put each triplet on a new line\n"
        "4. Return ONLY the triplets, no additional text\n"
        "5. If no triplets are found, return nothing (empty response)\n\n"
        "Example output format:\n"
        "Michael Corbat ; Citigroup ; employer\n"
        "VH1 ; Viacom ; owned_by"
    )

def parse_finred_response(response_text: str):
    """解析模型响应为triplet列表"""
    triplets = []
    lines = response_text.strip().split('\n')
    for line in lines:
        line = line.strip()
        if not line:
            continue
        if ';' in line:
            parts = [p.strip() for p in line.split(';')]
            if len(parts) == 3:
                # 验证关系类型是否在预定义列表中
                if parts[2] in RELATION_TYPES:
                    triplets.append({
                        "head": parts[0],
                        "tail": parts[1],
                        "relation": parts[2]
                    })
    return triplets

def ask_o3_mini_once(text):
    """调用o3-mini API"""
    url = f"{BASE_URL}/chat/completions"
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }
    
    user_text = _make_finred_prompt(text)
    
    payload = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": "You are a financial NLP expert specializing in relation extraction. Respond only with the requested triplet format."},
            {"role": "user", "content": user_text}
        ]
    }
    
    try:
        response = requests.post(url, headers=headers, json=payload, timeout=30)
        
        if response.status_code != 200:
            error_msg = f"API Error {response.status_code}: {response.text}"
            return [], "", False, error_msg
            
        result = response.json()
        
        if 'choices' not in result or len(result['choices']) == 0:
            return [], "", False, "No choices in response"
            
        txt = result['choices'][0]['message']['content']
        txt = _strip_code_fences(txt)
        triplets = parse_finred_response(txt)
        return triplets, txt, True, None
        
    except Exception as e:
        return [], "", False, str(e)

def ask_o3_mini(text):
    delay = 2.0
    for attempt in range(6):
        try:
            triplets, raw_response, success, error = ask_o3_mini_once(text)
            if success:
                return triplets, raw_response, True, None
            else:
                if attempt < 5:
                    time.sleep(delay)
                    delay = min(delay * 2, 60)
                    continue
                return [], raw_response, False, error
        except Exception as e:
            if attempt == 5:
                return [], "", False, str(e)
            time.sleep(delay)
            delay = min(delay * 2, 60)
    return [], "", False, "Exhausted retries"

# 测试一个样本
print("测试单个样本...")
test_sample = ds[0]
test_text = test_sample["text"]
print(f"测试文本: {test_text[:100]}...")

test_triplets, test_response, test_success, test_error = ask_o3_mini(test_text)
print(f"测试成功: {test_success}")
if test_success:
    print(f"识别的三元组: {test_triplets}")
    print(f"原始响应: {test_response[:200]}...")
    
    print("\n开始完整评估...")
    
    # 删除旧文件以重新运行
    if os.path.exists(pred_path):
        os.remove(pred_path)
    if os.path.exists(err_path):
        os.remove(err_path)
    
    rows_done = []
    err_rows = []
    buf = []
    save_every = 10

    total = len(ds)
    print(f"Starting o3-mini FINRED evaluation on {total} samples...")

    for i in tqdm(range(total)):
        x = ds[i]
        text = x["text"]
        gold_triplets = x.get("triplets", [])

        try:
            pred_triplets, raw_response, success, error_msg = ask_o3_mini(text)
            if not success:
                raise RuntimeError(error_msg)
        except Exception as e:
            pred_triplets = []
            raw_response = f"ERROR: {type(e).__name__}: {e}"
            err_rows.append({"row_idx": i, "error": raw_response, "text": text})
            success = False

        buf.append({
            "row_idx": i,
            "text": text[:200] + "..." if len(text) > 200 else text,
            "pred_raw": raw_response,
            "pred_triplets": json.dumps(pred_triplets),
            "gold_triplets": json.dumps(gold_triplets),
            "pred_count": len(pred_triplets),
            "gold_count": len(gold_triplets),
            "success": success
        })

        if len(buf) % save_every == 0:
            out = pd.DataFrame(rows_done + buf).sort_values("row_idx")
            out.to_csv(pred_path, index=False)
            if err_rows:
                pd.DataFrame(err_rows).to_csv(err_path, index=False)
            print(f"[checkpoint] saved {len(out)}/{total} -> {pred_path}")

    out = pd.DataFrame(rows_done + buf).sort_values("row_idx")
    out.to_csv(pred_path, index=False)
    if err_rows:
        pd.DataFrame(err_rows).to_csv(err_path, index=False)
    print(f"[done] o3-mini FINRED evaluation completed -> {pred_path}")
    
else:
    print(f"测试失败: {test_error}")

预测文件: /content/flare_finred_o3_mini_predictions.csv
错误文件: /content/flare_finred_o3_mini_errors.csv
测试单个样本...
测试文本: Wednesday, July 8, 2015 10:30AM IST (5:00AM GMT) Rimini Street Comment on Oracle Litigation Las Vega...
测试成功: True
识别的三元组: [{'head': 'Rimini Street, Inc.', 'tail': 'Las Vegas, United States', 'relation': 'headquarters_location'}, {'head': 'SAP AG', 'tail': 'Business Suite', 'relation': 'product_or_material_produced'}, {'head': 'SAP AG', 'tail': 'BusinessObjects', 'relation': 'product_or_material_produced'}, {'head': 'Oracle Corporation', 'tail': 'Siebel', 'relation': 'product_or_material_produced'}, {'head': 'Oracle Corporation', 'tail': 'PeopleSoft', 'relation': 'product_or_material_produced'}, {'head': 'Oracle Corporation', 'tail': 'JD Edwards', 'relation': 'product_or_material_produced'}, {'head': 'Oracle Corporation', 'tail': 'E-Business Suite', 'relation': 'product_or_material_produced'}, {'head': 'Oracle Corporation', 'tail': 'Oracle Database', 'relation': 'product_o

  1%|▋                                                                             | 10/1068 [01:46<2:59:14, 10.16s/it]

[checkpoint] saved 10/1068 -> /content/flare_finred_o3_mini_predictions.csv


  2%|█▍                                                                            | 20/1068 [03:05<2:19:48,  8.00s/it]

[checkpoint] saved 20/1068 -> /content/flare_finred_o3_mini_predictions.csv


...

[checkpoint] saved 1050/1068 -> /content/flare_finred_o3_mini_predictions.csv


 99%|███████████████████████████████████████████████████████████████████████████▍| 1060/1068 [2:38:35<00:59,  7.47s/it]

[checkpoint] saved 1060/1068 -> /content/flare_finred_o3_mini_predictions.csv


100%|████████████████████████████████████████████████████████████████████████████| 1068/1068 [2:39:54<00:00,  8.98s/it]

[done] o3-mini FINRED evaluation completed -> /content/flare_finred_o3_mini_predictions.csv


In [18]:
# Step 5: 分析 FLARE-FINRED 现有评估结果
import json
import pandas as pd
import os

print("="*60)
print("FLARE-FINRED 评估结果分析")
print("="*60)

# 读取评估结果
eval_path = "/content/flare_finred_o3_mini_evaluation_results.json"
if os.path.exists(eval_path):
    with open(eval_path, 'r') as f:
        results = json.load(f)
    
    print(f"\n数据集: {results['dataset']}")
    print(f"模型: {results['model']}")
    print(f"测试样本数: {results['total_samples']}")
    print(f"成功预测: {results['successful_predictions']}")
    print(f"失败预测: {results['failed_predictions']}")
    
    print(f"\n评估指标:")
    metrics = results.get('metrics', {})
    print(f"  Precision: {metrics.get('precision', 0):.4f}")
    print(f"  Recall: {metrics.get('recall', 0):.4f}")
    print(f"  F1-Score: {metrics.get('f1', 0):.4f}")
    print(f"  正确三元组数: {metrics.get('correct', 0)}/{metrics.get('total_gold', 0)}")
    
    # 检查是否有按关系类型的统计
    per_rel = results.get('per_relation_metrics', {})
    if per_rel:
        print(f"\n按关系类型统计 (前10):")
        sorted_rels = sorted(per_rel.items(), key=lambda x: x[1]['total'], reverse=True)
        for rel, stats in sorted_rels[:10]:
            print(f"  {rel}: {stats['correct']}/{stats['total']} (recall: {stats['recall']:.2%})")
    else:
        print(f"\n注意: 没有按关系类型的统计数据")
        print(f"原因: total_gold = {metrics.get('total_gold', 0)}，数据集中没有标注关系三元组")
    
    print(f"\n评估时间: {results['evaluation_time']}")
    
else:
    print(f"评估结果文件不存在: {eval_path}")
    print("请先运行 flare_finred 的评估代码")

FLARE-FINRED 评估结果分析

数据集: ChanceFocus/flare-finred
模型: o3-mini
测试样本数: 1068
成功预测: 1068
失败预测: 0

评估指标:
  Precision: 0.0000
  Recall: 0.0000
  F1-Score: 0.0000
  正确三元组数: 0/0

注意: 没有按关系类型的统计数据
原因: total_gold = 0，数据集中没有标注关系三元组

评估时间: 2026-03-21 02:56:53


In [17]:
# Step 5b: 生成总结报告
print("\n" + "="*60)
print("FLARE-FINRED 评估总结")
print("="*60)

# 从评估结果中提取关键信息
if os.path.exists(eval_path):
    with open(eval_path, 'r') as f:
        results = json.load(f)
    
    summary = {
        "dataset": "ChanceFocus/flare-finred",
        "model": "o3-mini",
        "total_samples": results['total_samples'],
        "successful_predictions": results['successful_predictions'],
        "total_gold_triplets": results['metrics']['total_gold'],
        "total_pred_triplets": results['metrics']['total_pred'],
        "precision": results['metrics']['precision'],
        "recall": results['metrics']['recall'],
        "f1": results['metrics']['f1'],
        "status": "completed" if results['metrics']['total_gold'] > 0 else "completed_no_labels",
        "note": "测试集无标注关系三元组" if results['metrics']['total_gold'] == 0 else "正常评估"
    }
    
    print(f"\n数据集: {summary['dataset']}")
    print(f"模型: {summary['model']}")
    print(f"总样本: {summary['total_samples']}")
    print(f"成功预测: {summary['successful_predictions']}")
    print(f"总Gold三元组: {summary['total_gold_triplets']}")
    print(f"总Pred三元组: {summary['total_pred_triplets']}")
    print(f"Precision: {summary['precision']:.4f}")
    print(f"Recall: {summary['recall']:.4f}")
    print(f"F1: {summary['f1']:.4f}")
    print(f"状态: {summary['status']}")
    print(f"备注: {summary['note']}")
    
    # 保存总结到文件
    summary_path = "/content/flare_finred_evaluation_summary.json"
    with open(summary_path, 'w') as f:
        json.dump(summary, f, indent=2)
    print(f"\n总结已保存到: {summary_path}")


FLARE-FINRED 评估总结

数据集: ChanceFocus/flare-finred
模型: o3-mini
总样本: 1068
成功预测: 1068
总Gold三元组: 0
总Pred三元组: 1440
Precision: 0.0000
Recall: 0.0000
F1: 0.0000
状态: completed_no_labels
备注: 测试集无标注关系三元组

总结已保存到: /content/flare_finred_evaluation_summary.json


In [19]:
# 重新分析 FLARE-FINRED 的预测结果
import json

# 读取评估结果
eval_path = "/content/flare_finred_o3_mini_evaluation_results.json"
with open(eval_path, 'r') as f:
    results = json.load(f)

print("="*60)
print("FLARE-FINRED 模型预测结果分析")
print("="*60)

print(f"\n数据集: {results['dataset']}")
print(f"模型: {results['model']}")
print(f"总样本数: {results['total_samples']}")
print(f"成功预测: {results['successful_predictions']}")
print(f"失败预测: {results['failed_predictions']}")

# 关键数据
print(f"\n模型预测的三元组总数: {results['metrics']['total_pred']}")
print(f"真实标注的三元组总数: {results['metrics']['total_gold']}")
print(f"正确预测的三元组数: {results['metrics']['correct']}")

print(f"\n评估指标:")
print(f"  Precision: {results['metrics']['precision']:.4f}")
print(f"  Recall: {results['metrics']['recall']:.4f}")
print(f"  F1-Score: {results['metrics']['f1']:.4f}")

print(f"\n说明:")
print(f"  - 模型预测了 {results['metrics']['total_pred']} 个三元组")
print(f"  - 但真实标注为 0，所以无法计算准确率")
print(f"  - 这可能是测试集本身没有标注（用于推理测试）")

FLARE-FINRED 模型预测结果分析

数据集: ChanceFocus/flare-finred
模型: o3-mini
总样本数: 1068
成功预测: 1068
失败预测: 0

模型预测的三元组总数: 1440
真实标注的三元组总数: 0
正确预测的三元组数: 0

评估指标:
  Precision: 0.0000
  Recall: 0.0000
  F1-Score: 0.0000

说明:
  - 模型预测了 1440 个三元组
  - 但真实标注为 0，所以无法计算准确率
  - 这可能是测试集本身没有标注（用于推理测试）


In [20]:
# 查找是否有保存了预测内容的文件
import os

print("\n查找预测内容文件...")

# 检查可能的文件位置
search_paths = ['/content', os.getcwd(), '.']

found_files = []
for search_path in search_paths:
    if os.path.exists(search_path):
        for file in os.listdir(search_path):
            if 'flare_finred' in file and ('pred' in file.lower() or 'output' in file.lower() or 'result' in file.lower()):
                if file.endswith('.csv') or file.endswith('.json') or file.endswith('.txt'):
                    full_path = os.path.join(search_path, file)
                    found_files.append(full_path)
                    print(f"找到: {full_path}")

if not found_files:
    print("未找到预测内容文件")
    print("\n当前目录下的相关文件:")
    for file in os.listdir('.'):
        if 'flare_finred' in file:
            print(f"  {file}")
    
    print("\n/content目录下的相关文件:")
    if os.path.exists('/content'):
        for file in os.listdir('/content'):
            if 'flare_finred' in file:
                print(f"  {file}")
                


查找预测内容文件...
找到: /content\flare_finred_o3_mini_evaluation_results.json


In [21]:
# 查看 evaluation_results.json 的完整结构
import json

eval_path = "/content/flare_finred_o3_mini_evaluation_results.json"
with open(eval_path, 'r') as f:
    results = json.load(f)

print("=== evaluation_results.json 的键 ===")
print(results.keys())

print("\n=== 完整内容 ===")
print(json.dumps(results, indent=2, ensure_ascii=False)[:3000])

=== evaluation_results.json 的键 ===
dict_keys(['model', 'dataset', 'split', 'total_samples', 'successful_predictions', 'failed_predictions', 'metrics', 'per_relation_metrics', 'evaluation_time'])

=== 完整内容 ===
{
  "model": "o3-mini",
  "dataset": "ChanceFocus/flare-finred",
  "split": "test",
  "total_samples": 1068,
  "successful_predictions": 1068,
  "failed_predictions": 0,
  "metrics": {
    "precision": 0.0,
    "recall": 0.0,
    "f1": 0.0,
    "correct": 0,
    "total_gold": 0,
    "total_pred": 1440
  },
  "per_relation_metrics": {},
  "evaluation_time": "2026-03-21 02:56:53"
}


In [22]:
# 如果 results 里有 predictions 字段，直接读取
import json

eval_path = "/content/flare_finred_o3_mini_evaluation_results.json"
with open(eval_path, 'r') as f:
    results = json.load(f)

# 检查是否有预测数据
if 'predictions' in results:
    print(f"找到 predictions 字段，共 {len(results['predictions'])} 条")
    print("\n第一条预测示例:")
    print(json.dumps(results['predictions'][0], indent=2, ensure_ascii=False))
else:
    print("没有 predictions 字段")
    print("\n可用的字段:")
    for key in results.keys():
        print(f"  - {key}: {type(results[key]).__name__}")
    
    # 如果有 per_sample_metrics 或其他可能包含预测的字段
    if 'per_sample_metrics' in results:
        print(f"\nper_sample_metrics 共 {len(results['per_sample_metrics'])} 条")
        print("第一条:", results['per_sample_metrics'][0] if results['per_sample_metrics'] else "空")

没有 predictions 字段

可用的字段:
  - model: str
  - dataset: str
  - split: str
  - total_samples: int
  - successful_predictions: int
  - failed_predictions: int
  - metrics: dict
  - per_relation_metrics: dict
  - evaluation_time: str


In [25]:
"""
================================================================================
FLARE-FINRED 评估结果说明
================================================================================

【评估概况】
- 数据集: ChanceFocus/flare-finred
- 模型: o3-mini
- 任务类型: 关系抽取 (Relation Extraction)
- 样本数量: 1,068

【执行情况】
- 成功处理: 1,068/1,068 (100%)
- 失败处理: 0
- 模型预测三元组总数: 1,440

【评估结果】
- Precision: 0.0000
- Recall: 0.0000
- F1-Score: 0.0000
- 正确三元组: 0/0

【问题说明】
指标为 0 的原因并非模型性能问题，而是数据集测试分片未提供标注答案：
- total_gold = 0: 测试集中没有真实标注的关系三元组
- total_pred = 1,440: 模型成功预测了 1,440 个关系三元组
- 由于无真实标签，无法计算 Precision/Recall/F1 等准确性指标

【结论】
✅ 模型运行成功：所有样本正常处理，模型能够提取关系三元组
⚠️ 无法评估准确性：测试集缺少标注，无法判断预测是否正确
📌 该情况不影响其他 FLARE 数据集的评估结果

【后续建议】
如需评估该数据集上的模型性能，建议：
1. 使用训练集或验证集（如果包含标注）
2. 或获取包含标注的测试集版本
3. 或在最终报告中说明该数据集为无标注推理测试集

================================================================================
"""

'\n================================================================================\nFLARE-FINRED 评估结果说明\n================================================================================\n\n【评估概况】\n- 数据集: ChanceFocus/flare-finred\n- 模型: o3-mini\n- 任务类型: 关系抽取 (Relation Extraction)\n- 样本数量: 1,068\n\n【执行情况】\n- 成功处理: 1,068/1,068 (100%)\n- 失败处理: 0\n- 模型预测三元组总数: 1,440\n\n【评估结果】\n- Precision: 0.0000\n- Recall: 0.0000\n- F1-Score: 0.0000\n- 正确三元组: 0/0\n\n【问题说明】\n指标为 0 的原因并非模型性能问题，而是数据集测试分片未提供标注答案：\n- total_gold = 0: 测试集中没有真实标注的关系三元组\n- total_pred = 1,440: 模型成功预测了 1,440 个关系三元组\n- 由于无真实标签，无法计算 Precision/Recall/F1 等准确性指标\n\n【结论】\n✅ 模型运行成功：所有样本正常处理，模型能够提取关系三元组\n⚠️ 无法评估准确性：测试集缺少标注，无法判断预测是否正确\n📌 该情况不影响其他 FLARE 数据集的评估结果\n\n【后续建议】\n如需评估该数据集上的模型性能，建议：\n1. 使用训练集或验证集（如果包含标注）\n2. 或获取包含标注的测试集版本\n3. 或在最终报告中说明该数据集为无标注推理测试集\n\n================================================================================\n'